# Chapter 7.3 - Padding and Stride

A convolution is not only defined by its kernel. Padding decides how the border is treated, and stride decides how densely the kernel samples locations. These choices control spatial resolution, information loss, computation, and the shape contracts between layers.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Every required tensor, helper, and model is defined inside this notebook. The cells use small tensors so that the mechanics are visible without turning the chapter into a larger experiment.

## You are done when you can

- explain padding and stride as architectural choices, not only formula inputs
- compute convolution output sizes as ordinary Python code
- verify padding and stride against `nn.Conv2d`
- explain why padding preserves border information
- debug impossible kernel/input geometry


In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def conv_out_size(input_size, kernel_size, padding=0, stride=1):
    return (input_size + 2 * padding - kernel_size) // stride + 1


## 7.3.0 The Problem This Notebook Solves

Chapter 7.2 used valid convolution: the kernel only visited positions where it fit fully inside the input. That made the output smaller. If we stack many such layers, spatial maps can shrink quickly.

Padding and stride are the two main controls introduced here:

- Padding adds border values before the kernel slides.
- Stride changes how far the kernel moves between windows.

These are not cosmetic settings. They change the representation pipeline:

```text
padding controls border treatment and spatial preservation
stride controls downsampling and compute
kernel size controls local receptive field
```

The key habit is to predict output shape before connecting layers. Shape formulas are not abstract math here; they are engineering contracts. If one layer produces an unexpected height or width, the next layer may fail or silently receive a representation different from what you intended.


## 7.3.1 Output Size Is a Mechanical Contract

For one spatial dimension, use this code pattern:

```text
add padding on both sides
subtract the kernel size
count stride steps that fit
include the first position
```

The formula is written as ordinary Python because the purpose is shape reasoning, not symbolic manipulation:

```python
out = (input_size + 2 * padding - kernel_size) // stride + 1
```

The integer division matters. With stride greater than 1, not every possible window start is visited. The output counts only starts where the kernel still fits.


In [ ]:
configs = [
    {"input_size": 8, "kernel_size": 3, "padding": 0, "stride": 1},
    {"input_size": 8, "kernel_size": 3, "padding": 1, "stride": 1},
    {"input_size": 8, "kernel_size": 3, "padding": 1, "stride": 2},
]

for cfg in configs:
    out = conv_out_size(**cfg)
    conv = nn.Conv2d(1, 1, cfg["kernel_size"], padding=cfg["padding"], stride=cfg["stride"])
    Y = conv(torch.zeros(1, 1, cfg["input_size"], cfg["input_size"]))
    print(cfg, "formula:", out, "pytorch:", shape(Y))
    assert shape(Y)[2:] == (out, out)


## 7.3.2 Padding Adds Artificial Border Values

Without padding, border pixels participate in fewer windows than center pixels. A corner pixel might appear in only one 3 by 3 valid window, while a center pixel appears in many. Padding changes that treatment.

Padding can preserve spatial size, which is useful when stacking layers because it prevents feature maps from shrinking after every convolution.

But padding is not new evidence. Zero padding adds artificial border values. The model can learn to handle them, but they are not real pixels. That is the tradeoff:

```text
padding preserves spatial dimensions and border participation
padding also introduces artificial boundary assumptions
```


In [ ]:
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
padded = torch.zeros(4, 4)
padded[1:3, 1:3] = X

print(padded)

conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
Y = conv(torch.zeros(1, 1, 5, 5))

assert shape(padded) == (4, 4)
assert shape(Y) == (1, 1, 5, 5)


## 7.3.3 Stride Skips Window Positions

Stride controls sampling density.

Stride 1 asks the local detector at every possible position. Stride 2 asks it at every other position. Larger stride reduces the spatial size of the feature map and reduces computation, but it also discards some location detail.

The theoretical meaning is downsampling:

```text
smaller spatial map
larger effective jump between neighboring output values
less memory and compute
less precise spatial information
```

Stride is therefore an architectural decision. It affects what information later layers can access.


In [ ]:
positions_stride_1 = list(range(0, 6 - 3 + 1, 1))
positions_stride_2 = list(range(0, 6 - 3 + 1, 2))

print("stride 1 starts:", positions_stride_1)
print("stride 2 starts:", positions_stride_2)

conv = nn.Conv2d(1, 1, kernel_size=3, stride=2)
Y = conv(torch.zeros(1, 1, 6, 6))

assert positions_stride_1 == [0, 1, 2, 3]
assert positions_stride_2 == [0, 2]
assert shape(Y) == (1, 1, 2, 2)


## 7.3.4 Padding and Stride Do Not Change Parameter Count

This is a key separation:

```text
parameter count: what weights are learned
output shape: where and how often those weights are applied
```

Padding and stride change the spatial geometry of the computation, but they do not create new kernel weights. A 3 by 3 kernel over 3 input channels with 8 output channels has the same learned weight count regardless of whether it scans densely, skips positions, or uses padding.

This helps separate capacity from resolution. More channels or larger kernels increase parameter count. Stride and padding mainly change feature-map size and border behavior.


In [ ]:
a = nn.Conv2d(3, 8, kernel_size=3, padding=0, stride=1)
b = nn.Conv2d(3, 8, kernel_size=3, padding=1, stride=2)

params_a = sum(p.numel() for p in a.parameters())
params_b = sum(p.numel() for p in b.parameters())

print("params without padding/stride:", params_a)
print("params with padding/stride:", params_b)

assert params_a == params_b == 8 * 3 * 3 * 3 + 8


## 7.3.5 Break It Deliberately: Kernel Too Large

The kernel must fit somewhere. If the input is 3 by 3 and the kernel is 5 by 5 with no padding, there is no valid local window.

The theory-level mistake is asking a local detector to inspect a neighborhood larger than the available representation. Padding could create artificial space around the input, but without padding, the operation has no valid output location.

This failure is useful because it connects the shape formula to a physical sliding-window interpretation.


In [ ]:
conv = nn.Conv2d(1, 1, kernel_size=5)
X = torch.zeros(1, 1, 3, 3)

try:
    conv(X)
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The kernel should not fit inside the input.")


## 7.3 Checkpoint

Answer these before moving on. Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. Why are padding and stride architectural choices rather than only API arguments?
2. What do padding and stride each control?
3. Why does `padding=1` preserve size for a 3 by 3 kernel with stride 1?
4. What information tradeoff does larger stride create?
5. Why does stride affect output shape but not parameter count?
6. What does it mean mechanically when the kernel is too large for the input?
